In [ ]:
%pip install pandas plotly google-generativeai


In [ ]:
dbutils.library.restartPython()


In [ ]:
"""
InsightForge AI — Anomaly Detection
=====================================
Detects statistical anomalies in any numeric column
using the IQR (Interquartile Range) method.

For each numeric column it computes:
- Q1, Q3, IQR boundaries
- Lower and upper fence values
- Rows that fall outside the fences
- Percentage of dataset affected

Also uses Gemini to explain why flagged rows
might be anomalous in business terms.

Author   : [Your Name]
Platform : Databricks
Model    : gemini-2.0-flash
"""

import pandas as pd
import plotly.express as px
import google.generativeai as genai

# ── Widgets ───────────────────────────────────────────────────
dbutils.widgets.text(
    "dataset_path",
    "/Volumes/insight/default/titanic/Titanic.csv",
    "Dataset Path"
)
dbutils.widgets.text("gemini_key", "", "Gemini API Key")

DATASET_PATH = dbutils.widgets.get("dataset_path")
GEMINI_KEY   = dbutils.widgets.get("gemini_key")
GEMINI_MODEL = "gemini-flash-latest"

# ── Load data ─────────────────────────────────────────────────
df = pd.read_csv(DATASET_PATH)

# ── Configure Gemini ──────────────────────────────────────────
genai.configure(api_key=GEMINI_KEY)
model = genai.GenerativeModel(GEMINI_MODEL)

print("=" * 55)
print("  InsightForge AI — Anomaly Detection")
print("=" * 55)
print(f"  Dataset  : {DATASET_PATH}")
print(f"  Shape    : {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  Columns  : {df.columns.tolist()}")
print(f"  Model    : {GEMINI_MODEL}")
print("=" * 55)


In [ ]:
def compute_iqr_bounds(series: pd.Series) -> dict:
    """
    Computes IQR-based anomaly detection boundaries
    for a single numeric column.

    The IQR method flags values that are more than
    1.5 times the interquartile range above Q3 or
    below Q1. This is the standard statistical method
    used by boxplots and most EDA tools.

    Parameters
    ----------
    series : pd.Series — a single numeric column

    Returns
    -------
    dict : Q1, Q3, IQR, lower fence, upper fence
    """
    q1  = series.quantile(0.25)
    q3  = series.quantile(0.75)
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    return {
        "q1"         : round(float(q1),          4),
        "q3"         : round(float(q3),          4),
        "iqr"        : round(float(iqr),         4),
        "lower_fence": round(float(lower_fence), 4),
        "upper_fence": round(float(upper_fence), 4),
    }


# Test on Age column
bounds = compute_iqr_bounds(df["Age"].dropna())

print("✅ compute_iqr_bounds() defined")
print()
print("IQR bounds for 'Age':")
for k, v in bounds.items():
    print(f"  {k:15} : {v}")
print()
print(f"  Interpretation:")
print(f"  Any Age below {bounds['lower_fence']} is unusually low")
print(f"  Any Age above {bounds['upper_fence']} is unusually high")


In [ ]:
def detect_column_anomalies(
    df : pd.DataFrame,
    col: str
) -> dict:
    """
    Detects anomalous rows in a single numeric column.

    Parameters
    ----------
    df  : pd.DataFrame — the full dataset
    col : str          — column name to check

    Returns
    -------
    dict containing:
        bounds        : IQR boundary values
        anomaly_mask  : boolean Series marking anomalous rows
        anomaly_count : number of anomalous rows
        anomaly_pct   : percentage of rows that are anomalous
        anomaly_rows  : DataFrame of anomalous rows
    """
    series = df[col].dropna()
    bounds = compute_iqr_bounds(series)

    # Flag rows outside the fences
    anomaly_mask = (
        (df[col] < bounds["lower_fence"]) |
        (df[col] > bounds["upper_fence"])
    )

    anomaly_rows  = df[anomaly_mask].copy()
    anomaly_count = int(anomaly_mask.sum())
    anomaly_pct   = round(anomaly_count / len(df) * 100, 2)

    return {
        "column"       : col,
        "bounds"       : bounds,
        "anomaly_mask" : anomaly_mask,
        "anomaly_count": anomaly_count,
        "anomaly_pct"  : anomaly_pct,
        "anomaly_rows" : anomaly_rows,
    }


# Test on Fare column
fare_result = detect_column_anomalies(df, "Fare")

print("✅ detect_column_anomalies() defined")
print()
print("Anomaly detection for 'Fare':")
print(f"  Lower fence   : {fare_result['bounds']['lower_fence']}")
print(f"  Upper fence   : {fare_result['bounds']['upper_fence']}")
print(f"  Anomalies     : {fare_result['anomaly_count']} rows")
print(f"  Percentage    : {fare_result['anomaly_pct']}%")
print()
print("Sample anomalous rows (highest fares):")
display(
    fare_result["anomaly_rows"]
    .sort_values("Fare", ascending=False)
    .head(5)
)


In [ ]:
def detect_anomalies(df: pd.DataFrame) -> tuple:
    """
    Runs anomaly detection across all numeric columns.

    Combines individual column results into:
    - A per-column report with counts and boundaries
    - A combined mask marking any row anomalous
      in at least one column
    - The full set of anomalous rows

    Parameters
    ----------
    df : pd.DataFrame — the dataset to analyse

    Returns
    -------
    tuple : (anomaly_report, anomalous_rows_df)
        anomaly_report   : dict of per-column results
        anomalous_rows_df: DataFrame of all flagged rows
    """
    numeric_cols  = df.select_dtypes(include="number").columns
    combined_mask = pd.Series(False, index=df.index)
    report        = {}

    print(f"Checking {len(numeric_cols)} numeric columns for anomalies...")
    print("─" * 55)

    for col in numeric_cols:
        result = detect_column_anomalies(df, col)
        report[col] = {
            "anomaly_count": result["anomaly_count"],
            "anomaly_pct"  : result["anomaly_pct"],
            "lower_fence"  : result["bounds"]["lower_fence"],
            "upper_fence"  : result["bounds"]["upper_fence"],
            "q1"           : result["bounds"]["q1"],
            "q3"           : result["bounds"]["q3"],
            "iqr"          : result["bounds"]["iqr"],
        }

        # Update combined mask — flag row if anomalous in ANY column
        combined_mask |= result["anomaly_mask"]

        # Print per-column summary
        bar = "█" * min(20, int(result["anomaly_pct"] / 2))
        print(
            f"  {col:15} : {result['anomaly_count']:4} anomalies"
            f"  ({result['anomaly_pct']:5.1f}%)  {bar}"
        )

    anomalous_rows = df[combined_mask].copy()

    print()
    print(f"  Total anomalous rows : {len(anomalous_rows)}")
    print(f"  Percentage of dataset: {round(len(anomalous_rows)/len(df)*100, 2)}%")
    print()
    print(f"✅ detect_anomalies() complete")

    return report, anomalous_rows


print("✅ detect_anomalies() defined")


In [ ]:
# Run on full Titanic dataset
anomaly_report, anomalous_rows = detect_anomalies(df)

print()
print("=" * 55)
print("ANOMALY REPORT — SUMMARY")
print("=" * 55)

# Build summary table
summary_data = []
for col, info in anomaly_report.items():
    summary_data.append({
        "Column"        : col,
        "Anomalies"     : info["anomaly_count"],
        "Anomaly %"     : info["anomaly_pct"],
        "Lower Fence"   : info["lower_fence"],
        "Upper Fence"   : info["upper_fence"],
        "IQR"           : info["iqr"],
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values("Anomaly %", ascending=False)
display(summary_df)


In [ ]:
def plot_anomaly_boxplot(df: pd.DataFrame, col: str) -> None:
    """
    Renders a boxplot for a single column showing
    the IQR fences and outlier points clearly.
    Points outside the whiskers are anomalies.

    Parameters
    ----------
    df  : pd.DataFrame — the dataset
    col : str          — column to visualise
    """
    fig = px.box(
        df,
        y        = col,
        title    = f"Boxplot — {col} (dots outside whiskers are anomalies)",
        template = "plotly_white",
        color_discrete_sequence = ["#4C72B0"],
        points   = "outliers"   # show only outlier points
    )
    fig.update_layout(
        yaxis_title = col,
        title_font  = dict(size=14),
        showlegend  = False
    )
    fig.show()


# Plot boxplots for columns with anomalies
cols_with_anomalies = [
    col for col, info in anomaly_report.items()
    if info["anomaly_count"] > 0
]

print(f"Plotting boxplots for {len(cols_with_anomalies)} columns with anomalies...")
print()

for col in cols_with_anomalies:
    plot_anomaly_boxplot(df, col)
    print(f"  ✅ {col} — {anomaly_report[col]['anomaly_count']} anomalies plotted")


In [ ]:
print("=" * 55)
print(f"ANOMALOUS ROWS ({len(anomalous_rows)} total)")
print("=" * 55)
print()
print("These rows have at least one value outside IQR fences:")
print()

# Show top anomalies sorted by Fare (most extreme first)
display(
    anomalous_rows
    .sort_values("Fare", ascending=False)
    .head(20)
)

print()
print("Top 5 highest fares (most extreme anomalies):")
print(
    anomalous_rows
    .nlargest(5, "Fare")
    [["PassengerId", "Name", "Pclass", "Fare", "Survived"]]
    .to_string(index=False)
)


In [ ]:
def explain_anomalies(
    df             : pd.DataFrame,
    anomaly_report : dict,
    anomalous_rows : pd.DataFrame
) -> str:
    """
    Asks Gemini to explain the anomalies in business terms.
    Receives the pre-computed anomaly report so Gemini
    does not need to do the statistical work — it only
    needs to interpret the findings.

    Parameters
    ----------
    df             : full dataset for context
    anomaly_report : dict from detect_anomalies()
    anomalous_rows : DataFrame of flagged rows

    Returns
    -------
    str : business explanation of the anomalies found
    """
    # Build anomaly summary for prompt
    anomaly_summary = []
    for col, info in anomaly_report.items():
        if info["anomaly_count"] > 0:
            anomaly_summary.append(
                f"  {col}: {info['anomaly_count']} anomalies "
                f"({info['anomaly_pct']}%) — "
                f"normal range: {info['lower_fence']} to {info['upper_fence']}"
            )

    prompt = f"""
You are a senior data analyst explaining anomalies to a business audience.

Dataset: Passenger records from the RMS Titanic, {df.shape[0]} rows.
Target variable: Survived (0=died, 1=survived)

ANOMALIES DETECTED (using IQR method):
{chr(10).join(anomaly_summary)}

SAMPLE ANOMALOUS ROWS (top 5 by Fare):
{anomalous_rows.nlargest(5, "Fare")[["PassengerId","Pclass","Fare","Age","Survived"]].to_string()}

TOTAL ANOMALOUS ROWS: {len(anomalous_rows)} out of {len(df)} ({round(len(anomalous_rows)/len(df)*100,1)}%)

Please explain:
1. WHAT ARE THESE ANOMALIES
   What do the flagged values represent in real-world terms?

2. WHY DO THEY EXIST
   Are these data errors or genuine extreme cases?

3. BUSINESS IMPACT
   How might these anomalies affect analysis or decisions?

4. RECOMMENDED ACTION
   Should these rows be removed, kept, or treated differently?

Keep the explanation practical and non-technical.
Use specific numbers from the anomaly report above.
"""

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Explanation unavailable: {str(e)}"


print("Asking Gemini to explain the anomalies...")
print("─" * 55)

explanation = explain_anomalies(df, anomaly_report, anomalous_rows)

print()
print("=" * 55)
print("GEMINI ANOMALY EXPLANATION")
print("=" * 55)
print()
print(explanation)


In [ ]:
# Save anomalous rows as CSV
anomaly_output_path = "/Volumes/insight/default/titanic/anomalous_rows.csv"
anomalous_rows.to_csv(anomaly_output_path, index=False)

# Save summary report as CSV
summary_output_path = "/Volumes/insight/default/titanic/anomaly_report.csv"
summary_df.to_csv(summary_output_path, index=False)

# Save Gemini explanation as text
explanation_path = "/Volumes/insight/default/titanic/anomaly_explanation.txt"
with open(explanation_path, "w") as f:
    f.write(explanation)

print("✅ Anomaly detection results saved:")
print(f"   Anomalous rows   : {anomaly_output_path}")
print(f"   Summary report   : {summary_output_path}")
print(f"   Gemini explains  : {explanation_path}")
print()

# Verify files exist
files = dbutils.fs.ls("/Volumes/insight/default/titanic/")
print("Files in Volume:")
for f in files:
    size_kb = round(f.size / 1024, 2)
    print(f"  {f.name:40} {size_kb:8} KB")


In [ ]:
def check_passenger(passenger_id: int) -> None:
    """
    Checks whether a specific passenger has anomalous values
    and shows which columns flagged them.

    Parameters
    ----------
    passenger_id : int — the PassengerId to look up
    """
    row = df[df["PassengerId"] == passenger_id]

    if row.empty:
        print(f"❌ PassengerId {passenger_id} not found")
        return

    print(f"Passenger {passenger_id}:")
    print(row.to_string(index=False))
    print()

    is_anomalous = passenger_id in anomalous_rows["PassengerId"].values

    if is_anomalous:
        print(f"⚠️  This passenger has anomalous values in:")
        for col, info in anomaly_report.items():
            val = row[col].values[0]
            if pd.isna(val):
                continue
            if val < info["lower_fence"] or val > info["upper_fence"]:
                print(
                    f"   {col}: {val} "
                    f"(normal range: {info['lower_fence']} "
                    f"to {info['upper_fence']})"
                )
    else:
        print(f"✅ This passenger has no anomalous values")


# Test on a few passengers
check_passenger(259)   # known high fare passenger
print()
check_passenger(1)     # regular passenger
